# Y · Gemma 4 E4B QLoRA on ControlSketch-Part

This notebook fine-tunes **`unsloth/gemma-4-E4B-it`** on the part-structured
SVG instruction dataset built by `training/prepare_dataset.py`. It targets
the **Gemma 4 Good Hackathon** (Future-of-Education + Unsloth tracks).

The output is a LoRA adapter the agent can drop into Ollama to serve the
edge-fine-tuned variant alongside the base E4B model.

**Runtime: free Kaggle T4 (~ 45–60 min for 2 epochs at batch=2 / seq=4096).**

Steps:
1. Install Unsloth (with Kaggle's pre-pinned `xformers` / `bitsandbytes`).
2. Load Gemma 4 E4B in 4-bit.
3. Wrap with QLoRA adapters (r=16, alpha=32).
4. Apply the Gemma 4 chat template to our `svg_lessons.jsonl`.
5. Train 2 epochs.
6. Sanity-check inference on a held-out chemistry prompt.
7. Save LoRA + (optional) push to HF + (optional) export GGUF.

**Why Gemma 4 E4B and not E2B?** The agent's vanilla "Edge" toolbar slot
already serves `gemma4:e4b`, so making the "Edge fine-tuned" slot E4B+LoRA
means the LoRA only has to add task quality on top of the same base — it
doesn't have to first claw back the gap from a smaller model. E4B in 4-bit
+ LoRA + activations at `batch=2 / seq=4096` lands at ~6–8 GB, comfortable
on a T4's 15 GB VRAM. Per Google's
[Gemma 4 launch post](https://blog.google/innovation-and-ai/technology/developers-tools/gemma-4/)
the family is E2B / E4B / 26B-MoE / 31B-Dense; Unsloth supports the whole
family day-one.


## 1 · Install dependencies

If running on Kaggle, enable **GPU T4 x2** in Settings → Accelerator (the
older single-T4 option has been retired; the second T4 will sit idle —
fine, this notebook is single-GPU).

**Why this install matters.** Kaggle's base image ships `transformers==5.0.0`,
which predates Gemma 4 support — its config registry doesn't know
`model_type: "gemma4"`. The fix is to pin **`transformers==5.5.0`** (the
first stable release with Gemma 4 support, and what Unsloth's own
[Gemma 4 E4B vision notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E4B)-Vision.ipynb)
uses) plus a small set of dep floors. We mirror Unsloth's official install
verbatim — they've already de-risked the version matrix for this exact
model.

After this cell finishes, **click Run → Restart & Run All** once so the
kernel re-imports the new `transformers` cleanly.


In [2]:
%%capture
import os, re

# Kaggle / Colab both have torch preinstalled — match xformers to it.
import torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = "xformers==" + {
    "2.10": "0.0.34",
    "2.9":  "0.0.33.post1",
    "2.8":  "0.0.32.post2",
}.get(v, "0.0.34")

# Pass 1: install the dependency floor (with resolver).
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer

# Pass 2: install bleeding-edge code without disturbing the floor.
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps "transformers==5.5.0" "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
!pip install --no-deps --upgrade timm   # Gemma 4 vision/audio backbone

torch._dynamo.config.recompile_limit = 64

print("Install done. Click Run -> Restart & Run All so the kernel re-imports transformers.")


### 1a · Verify the upgrade actually landed

Run this **after the kernel restart**, before importing Unsloth. You
should see `transformers  5.5.x` and `huggingface-hub  0.3x` or newer.
If `transformers` still shows `5.0.0`, the upgrade didn't apply —
re-run cell 1 and restart again.


In [3]:
import importlib.metadata as md

for pkg in [
    "transformers",
    "huggingface-hub",
    "tokenizers",
    "unsloth",
    "unsloth_zoo",
    "trl",
    "accelerate",
    "peft",
    "bitsandbytes",
    "datasets",
    "timm",
    "torchao",
]:
    try:
        print(f"{pkg:>18}  {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:>18}  NOT INSTALLED")


      transformers  5.5.0
   huggingface-hub  1.15.0
        tokenizers  0.22.2
           unsloth  2026.5.2
       unsloth_zoo  2026.5.2
               trl  1.4.0
        accelerate  1.13.0
              peft  0.19.1
      bitsandbytes  0.49.2
          datasets  4.3.0
              timm  1.0.27
           torchao  0.17.0


## 2 · Load Gemma 4 E4B in 4-bit

We load through **`FastVisionModel`** (not `FastLanguageModel` /
`FastModel`) for one specific reason: at inference time the agent feeds
the model a **PNG snapshot of the student's whiteboard** along with the
text prompt. Loading via `FastLanguageModel` would either drop the
vision tower or skip its weights — meaning the LoRA we save and the
GGUF we export would be language-only with no path to do `model.generate(image, prompt)`.
We'd ship a fine-tuned tutor that can't read the canvas.

This matches Unsloth's official
[Gemma 4 E4B vision notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E4B)-Vision.ipynb)
exactly (cell 7 there).

Unsloth aliases `unsloth/gemma-4-E4B-it` to its pre-quantised mirror
`unsloth/gemma-4-e4b-it-unsloth-bnb-4bit` — that's expected and what we
want. If the load fails, try one of:

* `unsloth/gemma-4-e4b-it-unsloth-bnb-4bit`  (the real underlying tag)
* `unsloth/gemma-4-e4b-it`                   (plain mirror)
* `google/gemma-4-E4B-it`                    (raw weights; Unsloth quantises on the fly)

`use_gradient_checkpointing="unsloth"` is set at load time (Unsloth's
custom impl, faster and lighter than the stock variant). It's the main
reason E4B + LoRA + activations fit on T4 at our seq length.

If you hit OOM, drop to `batch=1 / grad_accum=8` in cell 5 — effective
batch stays 8.


In [24]:
import os
import torch
from huggingface_hub import hf_hub_download
from unsloth import FastVisionModel

# The model exists on HF, but Kaggle/Hub version mismatches can make
# Unsloth report "No config file found". Probe HF first so failures are
# explicit and so we can use the exact repo Kaggle can see.
MODEL_CANDIDATES = [
    "unsloth/gemma-4-e4b-it-unsloth-bnb-4bit",  # pre-quantized mirror; fastest path
    "unsloth/gemma-4-E4B-it",                   # Unsloth base alias
    "google/gemma-4-E4B-it",                    # raw Google weights; may need HF access approval
]

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
MODEL_ID = None
last_error = None

for candidate in MODEL_CANDIDATES:
    try:
        path = hf_hub_download(
            repo_id=candidate,
            filename="config.json",
            token=hf_token,
            force_download=False,
        )
        print(f"Using {candidate}; config -> {path}")
        MODEL_ID = candidate
        break
    except Exception as exc:
        last_error = exc
        print(f"Could not access {candidate}: {type(exc).__name__}: {exc}")

if MODEL_ID is None:
    raise RuntimeError(
        "Could not access any Gemma 4 E4B model config from Hugging Face. "
        "Check Kaggle Internet=On, then optionally set HF_TOKEN if the repo "
        "requires gated access."
    ) from last_error

MAX_SEQ_LEN = 4096

model, processor = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
    token = hf_token,
    use_exact_model_name = True,
    device_map = {"": 0},
)

import torch
torch.cuda.set_device(0)
torch.cuda.empty_cache()


Using unsloth/gemma-4-e4b-it-unsloth-bnb-4bit; config -> /root/.cache/huggingface/hub/models--unsloth--gemma-4-e4b-it-unsloth-bnb-4bit/snapshots/91872ae26ac6bfbae6ee6a8acdc73d3370d1d227/config.json
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

## 3 · Add QLoRA adapters — language layers only

LoRA targets need a careful pick. Even though we loaded the vision tower
in cell 2 (so it's available at inference), our **training** data is
text-only — every row in `svg_lessons.jsonl` is `{user: "Draw a rabbit,
decomposing into named parts.", assistant: "[draw_part: ...] ..."}`,
zero images. Two consequences:

1. **`finetune_vision_layers=False`** — there are no image inputs in
   our dataset, so there are no real gradients to flow through the
   vision encoder. Turning this on would either leave a wasted LoRA
   shell or, worse, corrupt Google's pretrained vision weights from
   spurious gradient noise.
2. **`finetune_audio_layers=False`** — same reason; we have no audio.
3. **`finetune_language_layers=True` + attention + MLP** — this is
   where the new behaviour actually has to land: the
   `[draw_part: ...] ... [/draw_part]` block syntax, valid path-data
   sequences (`M L H V C S Q T A Z`), pedagogical decomposition. All of
   it is sequence modelling, so all of it lives in the language tower.

Put differently: SVG is text. The model emits `<path d="M 200 80 ...">`
as token sequences; the OUTPUT renders to pixels somewhere downstream
but the model itself never sees those pixels. Vision capability stays
relevant for *reading* the student's canvas at inference (where it's
already pretrained and excellent), not for *writing* SVG.

Hyper-params follow the plan: `r=16, α=32, dropout=0`. We keep
`target_modules="all-linear"` per Unsloth's recipe — they handle picking
the right linear projections inside the language tower for us.


In [26]:
from peft import PeftModel

if isinstance(model, PeftModel) or hasattr(model, "peft_config"):
    print("LoRA adapters already attached; skipping get_peft_model.")
else:
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers     = False,   # no image inputs in our dataset
        finetune_audio_layers      = False,   # no audio inputs in our dataset
        finetune_language_layers   = True,
        finetune_attention_modules = True,
        finetune_mlp_modules       = True,

        r = 16,
        lora_alpha = 32,
        lora_dropout = 0,
        bias = "none",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
        target_modules = "all-linear",
    )

## 4 · Apply chat template + load `svg_lessons.jsonl`

Mount your prepared JSONL by either uploading it as a Kaggle Dataset (e.g.
`y-svg-lessons`) or by setting `DATASET_PATH` to the HF dataset id you
pushed via `prepare_dataset.py --push-hf …`.


In [27]:
from unsloth import get_chat_template
from datasets import load_dataset
from pathlib import Path

# Gemma 4 ships a fresh chat template. If your installed unsloth is older
# and doesn't know "gemma-4" yet, fall back to "gemma-3" (same
# `<start_of_turn>user/model<end_of_turn>` markers, training works fine).
try:
    processor = get_chat_template(processor, "gemma-4")
    print("Retrieved Gemma-4 template")
except Exception:
    processor = get_chat_template(processor, "gemma-3")

# Default Kaggle path (after `Add Data → y-svg-lessons`):
KAGGLE_PATH = Path("/kaggle/input/datasets/abhijithscience/y-svg-lessons/svg_lessons.jsonl")
LOCAL_PATH = Path("svg_lessons.jsonl")

if KAGGLE_PATH.exists():
    ds = load_dataset("json", data_files=str(KAGGLE_PATH), split="train")
elif LOCAL_PATH.exists():
    ds = load_dataset("json", data_files=str(LOCAL_PATH), split="train")
else:
    ds = load_dataset(HF_DATASET_ID, split="train")

print(f"loaded {len(ds)} rows; first row keys: {list(ds[0].keys())}")


Retrieved Gemma-4 template
loaded 400 rows; first row keys: ['messages']


In [28]:
# Apply the chat template ourselves (text path) so we can use the regular
# SFTTrainer instead of UnslothVisionDataCollator. Our dataset is text-only
# (no images), so this is the cleaner path even though we loaded the model
# through FastVisionModel.

SHORT_SYSTEM = """You are Y, an SVG-native whiteboard tutor. Reply only with the allowed tool tags.
Use concise [text:] captions and decompose diagrams into [draw_part: ...] blocks.
Use valid SVG path data and close every draw_part block with [/draw_part]."""

def format_row(example):
    msgs = example["messages"]

    user_msg = next(m for m in msgs if m["role"] == "user")
    assistant_msg = next(m for m in msgs if m["role"] == "assistant")

    short_messages = [
        {"role": "system", "content": SHORT_SYSTEM},
        {"role": "user", "content": user_msg["content"]},
        {"role": "assistant", "content": assistant_msg["content"]},
    ]

    text = processor.apply_chat_template(
        short_messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

ds = ds.map(format_row, num_proc=2)

sample = ds[0]["text"]
print(sample[:1000])
print("...")
print(sample[-800:])
print("token_len:", len(processor.tokenizer(sample).input_ids))

Map (num_proc=2):   0%|          | 0/400 [00:00<?, ? examples/s]

<bos><|turn>system
You are Y, an SVG-native whiteboard tutor. Reply only with the allowed tool tags.
Use concise [text:] captions and decompose diagrams into [draw_part: ...] blocks.
Use valid SVG path data and close every draw_part block with [/draw_part].<turn|>
<|turn>user
Draw a rabbit, decomposing into named parts.<turn|>
<|turn>model
[text: "Let's draw a rabbit."]
[text: "Drawing the rounded head facing left."]
[draw_part: name="rounded head facing left" viewBox="0 0 512 512"]
M 185 250 C 184 204 199 232 201 182
M 279 258 C 330 202 256 229 294 151
M 195 244 C 229 261 177 286 185 227
M 263 155 C 202 169 200 192 207 178
M 264 195 C 228 264 278 136 244 213
M 204 262 C 268 278 270 260 244 252
M 283 252 C 221 287 183 250 224 267
M 249 257 C 276 263 278 262 258 265
[/draw_part]
[text: "Drawing the two long upright ears."]
[draw_part: name="two long upright ears" viewBox="0 0 512 512"]
M 325 89 C 316 176 291 116 285 199
M 317 127 C 331 63 341 36 289 82
M 262 160 C 216 152 255 197 228 74

In [29]:
sample = ds[0]["text"]
print(sample.find("<start_of_turn>model\n"))
print(len(processor.tokenizer(sample).input_ids))

-1
1394


## 5 · Train

Plan hyper-params: `lr=2e-4`, `batch=2`, `grad_accum=4`, `epochs=2`. With
~400 rows that's roughly 100 optimiser steps — small but enough to teach the
`[draw_part]` block-output structure on top of an already-instruction-tuned
base.

We set `train_on_responses_only` so the loss is only computed on the
assistant tokens; the long system prompt is masked out. The trainer uses
`processor.tokenizer` (a plain text tokenizer) under the hood — the
vision tower is just along for the ride and gets no gradient updates
because `finetune_vision_layers=False` and there are no image tensors
in the batch.


In [30]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import inspect

INSTRUCTION_PART = "<|turn>user\n"
RESPONSE_PART = "<|turn>model\n"

sample = ds[0]["text"]
assert INSTRUCTION_PART in sample, f"instruction marker not found in: {sample[:200]!r}"
assert RESPONSE_PART in sample, f"response marker not found in: {sample[:200]!r}"
print("Markers found at:", sample.find(INSTRUCTION_PART), sample.find(RESPONSE_PART))

sft_params = inspect.signature(SFTConfig.__init__).parameters
length_kwargs = {}
if "max_seq_length" in sft_params:
    length_kwargs["max_seq_length"] = MAX_SEQ_LEN
elif "max_length" in sft_params:
    length_kwargs["max_length"] = MAX_SEQ_LEN
if "packing" in sft_params:
    length_kwargs["packing"] = False
print("SFT length kwargs:", length_kwargs)

trainer = SFTTrainer(
    model=model,
    processing_class=processor.tokenizer,
    train_dataset=ds,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        max_steps=-1,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_strategy="no",
        **length_kwargs,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_PART,
    response_part=RESPONSE_PART,
)

trainer_stats = trainer.train()

Markers found at: 265 329
SFT length kwargs: {'max_seq_length': 4096, 'packing': False}


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/400 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/400 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.091540
10,0.822612
15,0.456087
20,0.383153
25,0.356331
30,0.340121
35,0.331271
40,0.324867
45,0.324713
50,0.315986


## 6 · Sanity-check inference

Generate one short response. With LoRA loaded, the model should emit our
`[text:]` + `[draw_part: ...] ... [/draw_part]` block format.


In [35]:
import torch
from transformers import TextStreamer
import time

system_prompt = SHORT_SYSTEM if "SHORT_SYSTEM" in globals() else (
    "You are Y, an SVG-native whiteboard tutor. Reply only with the allowed tool tags. "
    "Use concise [text:] captions and decompose diagrams into [draw_part: ...] blocks. "
    "Use valid SVG path data and close every draw_part block with [/draw_part]."
)
test_user = "Draw a benzene ring with alternating bonds, decomposing into named parts."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_user},
]

input_text = processor.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor.tokenizer(
    input_text,
    return_tensors="pt",
).to(model.device)

FastVisionModel.for_inference(model)

time1 = time.time()
streamer = TextStreamer(processor.tokenizer, skip_prompt=True)
with torch.no_grad():
    _ = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.95,
        top_k=64,
        streamer=streamer,
    )
time2 = time.time()

print("Time taken for response generation: {} s".format(time2 - time1))

[text:="A benzene ring with alternating bonds"]
[draw_part: "The six carbon atoms forming the ring"]
<svg width="300" height="300" viewBox="0 0 300 300" xmlns="http://www.w3.org/2000/svg">
  <g id="benzene ring carbons">
    <circle cx="150" cy="100" r="10" fill="black"/>
    <circle cx="200" cy="150" r="10" fill="black"/>
    <circle cx="250" cy="200" r="10" fill="black"/>
    <circle cx="100" cy="220" r="10" fill="black"/>
    <circle cx="50" cy="180" r="10" fill="black"/>
    <circle cx="180" cy="250" r="10" fill="black"/>
  </g>
</svg>
[/draw_part]
[draw_part: "The alternating bonds within the ring"]
<svg width="300" height="300" viewBox="0 0 300 300" xmlns="http://www.w3.org/2000/svg">
  <g id="alternating bonds">
    <!-- Bonds connecting C1-C2, C2-C3, C3-C4, C4-C5, C5-C6, C6-C1 -->
    <path d="M 150 100 Q 200 150 250 200" stroke="black" stroke-width="2" fill="none"/>
    <path d="M 200 150 Q 250 200 100 220" stroke="black" stroke-width="2" fill="none"/>
    <path d="M 100 220 Q

## 7 · Save LoRA adapter

Local save (relative to the notebook's working dir). Kaggle keeps `Outputs/`
between sessions and lets you download the artefact. We save the
`processor` (which carries both the image processor and the tokenizer)
so the merged artefact stays multimodal — ready to read whiteboard PNGs
at inference time.


In [32]:
LORA_DIR = "y-gemma4-svg-lora"
model.save_pretrained(LORA_DIR)
processor.save_pretrained(LORA_DIR)
print("saved adapter →", LORA_DIR)


Unsloth: Restored added_tokens_decoder metadata in y-gemma4-svg-lora/tokenizer_config.json.


saved adapter → y-gemma4-svg-lora


### 7a · Push to Hugging Face (optional)

Uncomment after running `from huggingface_hub import login; login("HF_TOKEN")`.


In [37]:
from huggingface_hub import login
login()
HF_REPO = "QuantumTransformer/y-gemma4-svg-lora"
model.push_to_hub(HF_REPO, token=True)
processor.push_to_hub(HF_REPO, token=True)
print("pushed →", HF_REPO)


README.md:   0%|          | 0.00/580 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/QuantumTransformer/y-gemma4-svg-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmppufb8hs6/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

pushed → QuantumTransformer/y-gemma4-svg-lora


## 8 · GGUF export for Ollama (optional)

Saves a single `q4_k_m` GGUF that the next phase (`models/Modelfile.y-gemma4`)
will reference for `ollama create y-gemma4 -f ...`.

Gemma 4 has day-one llama.cpp support, so the export should "just work" on
recent llama.cpp builds. If it fails on Kaggle's pinned `llama-cpp-python`,
ship the LoRA + dataset on HF as the primary Unsloth-track deliverable and
let users merge to GGUF locally with `unsloth.save.save_to_gguf`.


In [ ]:
# model.save_pretrained_gguf(
#     "y-gemma4-svg-q4_k_m",
#     processor.tokenizer,
#     quantization_method="q4_k_m",
# )
# print("wrote y-gemma4-svg-q4_k_m/y-gemma4-svg-q4_k_m.gguf")


## 9 · Reproducibility

* Base model: `unsloth/gemma-4-E4B-it` (~ 4 B effective params during
  inference, loaded 4-bit). Per Google's
  [Gemma 4 launch post](https://blog.google/innovation-and-ai/technology/developers-tools/gemma-4/)
  E4B is the larger of the two edge-targeted Gemma 4 variants and matches
  the vanilla "Edge" tag (`gemma4:e4b`) the agent serves by default.
* Loaded through `FastVisionModel` (multimodal entry point) so the
  vision tower is preserved end-to-end. The agent passes whiteboard
  PNGs at inference; the saved adapter + processor remain multimodal.
* Dataset: 400 rows of `duxiaodan/ControlSketch-Part`, distilled by
  `training/prepare_dataset.py` into `[text:] / [draw_part: …] / [/draw_part]`
  blocks. Text-only — no image inputs in the training data.
* QLoRA: r=16, α=32, dropout=0, no bias.
  * `finetune_vision_layers=False`, `finetune_audio_layers=False` —
    no image / audio gradients available, so we don't perturb the
    pretrained encoders.
  * `finetune_language_layers=True`, `finetune_attention_modules=True`,
    `finetune_mlp_modules=True` — this is where the new
    `[draw_part]` block-output behaviour lives.
  * `target_modules="all-linear"` per Unsloth's recipe.
* Optimiser: AdamW-8-bit, lr=2e-4, linear schedule, 10 warmup steps,
  weight-decay=0.01.
* Training: 2 epochs (~ 100 steps) on a free T4 (~45–60 min).
* Eval: qualitative — the held-out benzene prompt should emit at least
  two well-formed `[draw_part]` blocks with sane viewBox and ≤ 8 path
  commands per part. Run `api/scripts/smoke_drawpart.py` against the
  re-served Ollama model to confirm.

The plan calls this the "Unsloth track" deliverable; the LoRA + the
prepared dataset on HF, both linked from the writeup, satisfy the track
requirements even if the GGUF export fails.
